In [ ]:
import polars as pl
from tcrtrifold.tcrdock_utils import (
    dgeom_ndarr_from_dgeom_series,
    mn_distr_from_dgeom_ndarr,
    mn_distance_from,
    un_cossin_embed,
)
from tcrtrifold.utils import filter_to_cog_thresh, FORMAT_ANTIGEN_COLS, FORMAT_TCR_COLS


iedb_II_conf = pl.read_parquet(
    "../../data/iedb_II/triad/iedb_II_triad.conf_af3.parquet"
)
iedb_II_tcrdock = pl.read_parquet(
    "../../data/iedb_II/triad/staged/iedb_II_triad.af3_tcrdock.parquet"
)

iedb_II = iedb_II_conf.join(
    iedb_II_tcrdock.select("pred_dgeom_4", "job_name").with_columns(
        pl.col("pred_dgeom_4").struct.unnest()
    ),
    on="job_name",
    how="inner",
)

template_dgeom = pl.read_csv(
    "../../data/pdb/raw/ternary_templates_v2.tsv", separator="\t"
).with_columns(
    pl.struct(
        **{
            k: pl.col(k)
            for k in [
                "d",
                "torsion",
                "mhc_unit_x_is_negative",
                "tcr_unit_y",
                "tcr_unit_z",
                "tcr_unit_x_is_negative",
                "mhc_unit_y",
                "mhc_unit_z",
            ]
        }
    ).alias("dgeom"),
    pl.when(pl.col("mhc_class") == 1)
    .then(pl.lit("I"))
    .otherwise(pl.lit("II"))
    .alias("mhc_class"),
)

class_II_t_dgeom = dgeom_ndarr_from_dgeom_series(
    template_dgeom.filter(pl.col("mhc_class") == "II").select("dgeom").to_series()
)

class_II_distr = mn_distr_from_dgeom_ndarr(class_II_t_dgeom)

_, iedb_II_p_dgeom = mn_distance_from(
    dgeom_ndarr_from_dgeom_series(iedb_II.select("pred_dgeom_4").to_series()),
    *class_II_distr,
)

iedb_II = iedb_II.with_columns(pl.Series(name="p_dgeom", values=iedb_II_p_dgeom))

iedb_II = iedb_II.explode("references")

In [12]:
from tcrtrifold.utils import FORMAT_ANTIGEN_COLS
from tcrtrifold.eval_utils import antigen_raw_score_auc
import sklearn.metrics as metrics
from scipy.stats import mannwhitneyu, norm, false_discovery_control
import numpy as np
import matplotlib.pyplot as plt


def plot_volcano(
    df, featnames, feat_type, grouping_cols=FORMAT_ANTIGEN_COLS + ["references"]
):

    antigen_st = df.filter(pl.col("cognate")).select(grouping_cols).unique()

    feat_df = []

    for feat, ft in zip(featnames, feat_type):

        for row in antigen_st.iter_rows(named=True):
            focal_a_st = pl.DataFrame([row]).select(pl.exclude("job_name"))
            focal_pos = df.join(focal_a_st, on=grouping_cols)
            focal_neg = df.join(
                focal_pos.select(FORMAT_ANTIGEN_COLS).unique(),
                on=FORMAT_ANTIGEN_COLS,
            ).filter(~pl.col("cognate"))

            focal_triad = pl.concat([focal_pos, focal_neg])

            dat = focal_triad.select(feat, "cognate").to_numpy()

            fpr, tpr, threshold = metrics.roc_curve(dat[:, 1], dat[:, 0])
            # roc_auc = abs(metrics.auc(fpr, tpr) - 0.5) + 0.5
            roc_auc = metrics.auc(fpr, tpr)

            feat_df.append(
                {
                    "auc": roc_auc,
                    "fpr": list(fpr),
                    "tpr": list(tpr),
                    "featname": feat,
                    "feat_type": ft,
                }
            )

    feat_df = pl.DataFrame(feat_df)

    return feat_df


docking_feats = [
    "d",
    "mhc_unit_y",
    "mhc_unit_z",
    "tcr_unit_y",
    "tcr_unit_z",
    "torsion",
    "p_dgeom",
]

interface_feats = [
    "mean_p_tcr_interface_pae",
    "mean_tcr_pmhc_interface_pae",
    "mean_p_tcr_interface_contact_prob",
    "mean_tcr_pmhc_interface_contact_prob",
    "mean_p_tcr_pae",
    "mean_tcr_p_pae",
    "mean_mhc_tcr_pae",
    "mean_tcr_mhc_pae",
    "mean_p_mhc_pae",
    "tcr_mhc_contacts",
    "peptide_tcr_contacts",
]


local_feats = [
    "peptide_mean_pLDDT",
    "tcr_1_cdr_1_mean_pLDDT",
    "tcr_1_cdr_2_mean_pLDDT",
    "tcr_1_cdr_2_5_mean_pLDDT",
    "tcr_1_cdr_3_mean_pLDDT",
    "tcr_2_cdr_1_mean_pLDDT",
    "tcr_2_cdr_2_mean_pLDDT",
    "tcr_2_cdr_2_5_mean_pLDDT",
    "tcr_2_cdr_3_mean_pLDDT",
    "tcr_cdrs_mean_pLDDT",
    "mhc_helices_mean_pLDDT",
]

summary_feats = [
    "iptm",
    "ptm",
    "ranking_score",
]

featnames = docking_feats + interface_feats + local_feats + summary_feats
feat_type = (
    ["docking"] * len(docking_feats)
    + ["interface"] * len(interface_feats)
    + ["local"] * len(local_feats)
    + ["summary"] * len(summary_feats)
)


df = plot_volcano(iedb_II, featnames, feat_type)

In [13]:
df.group_by("featname").agg(pl.col("auc").mean()).sort(by="auc", descending=True)

featname,auc
str,f64
"""ranking_score""",0.669175
"""mhc_helices_mean_pLDDT""",0.668817
"""iptm""",0.668698
"""ptm""",0.668075
"""peptide_mean_pLDDT""",0.665959
…,…
"""mean_p_mhc_pae""",0.331816
"""mean_mhc_tcr_pae""",0.328994
"""mean_tcr_p_pae""",0.3269


In [14]:
import polars as pl
from tcrtrifold.tcrdock_utils import (
    dgeom_ndarr_from_dgeom_series,
    mn_distr_from_dgeom_ndarr,
    mn_distance_from,
    un_cossin_embed,
)
from tcrtrifold.utils import filter_to_cog_thresh, FORMAT_ANTIGEN_COLS, FORMAT_TCR_COLS


iedb_I_conf = pl.read_parquet(
    "../../data/iedb_I/triad/staged/iedb_I_triad.conf_af3.parquet"
)
iedb_I = iedb_I_conf
# iedb_I_tcrdock = pl.read_parquet(
#     "../../data/iedb_I/triad/staged/iedb_I_triad.af3_tcrdock.parquet"
# )

# iedb_I = iedb_II_conf.join(
#     iedb_I_tcrdock.select("pred_dgeom_4", "job_name").with_columns(
#         pl.col("pred_dgeom_4").struct.unnest()
#     ),
#     on="job_name",
#     how="inner",
# )

# class_I_t_dgeom = dgeom_ndarr_from_dgeom_series(
#     template_dgeom.filter(pl.col("mhc_class") == "I").select("dgeom").to_series()
# )

# class_I_distr = mn_distr_from_dgeom_ndarr(class_I_t_dgeom)

# _, iedb_I_p_dgeom = mn_distance_from(
#     dgeom_ndarr_from_dgeom_series(iedb_II.select("pred_dgeom_4").to_series()),
#     *class_I_distr,
# )

# iedb_I = iedb_II.with_columns(pl.Series(name="p_dgeom", values=iedb_I_p_dgeom))

iedb_I = iedb_I.explode("references")

In [16]:
featnames = interface_feats + local_feats + summary_feats
feat_type = (
    ["interface"] * len(interface_feats)
    + ["local"] * len(local_feats)
    + ["summary"] * len(summary_feats)
)


df = plot_volcano(iedb_I, featnames, feat_type)

In [17]:
df.group_by("featname").agg(pl.col("auc").mean()).sort(by="auc", descending=True)

featname,auc
str,f64
"""ptm""",0.650301
"""ranking_score""",0.649836
"""iptm""",0.649663
"""mean_tcr_pmhc_interface_contac…",0.647008
"""peptide_tcr_contacts""",0.64376
…,…
"""mean_tcr_mhc_pae""",0.346695
"""mean_mhc_tcr_pae""",0.344657
"""mean_tcr_p_pae""",0.342634
